# Backend experiment: open-langgraph-platform

[`open-langgraph-platform`](https://github.com/HyunjunJeon/open-langgraph-platform) is another open-source self-hosted LangGraph Platform reimplementation (FastAPI + Postgres + Redis). It has no PyPI package, so it's vendored as a git submodule (a fork with local fixes for upstream migration/auth/dependency bugs -- see README.md's \"Known upstream issues\"), run here via Docker.

This notebook is a self-contained, reproducible experiment: it starts the `open-langgraph` backend, deploys the three example agents (`researcher`, `coder`, `reviewer`), calls each one directly through `langgraph_sdk`, then runs the `supervisor` agent locally and has it orchestrate all three **via `RemoteGraph`**.


In [1]:
import sys

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")

from langgraph_sdk import get_sync_client

from remotegraph.backends.open_langgraph import OpenLangGraphBackend as Backend

backend = Backend()
print(backend.name, "->", backend.base_url)


open-langgraph -> http://127.0.0.1:8001


## 1. Start the backend

`backend.up()` runs `docker compose up -d --build` (postgres + redis + the app, built from `vendor/open-langgraph-platform`). First run builds the image, so this can take a minute or two.

In [2]:
SERVED_AGENTS = {
    "researcher": "agents/researcher/graph.py:graph",
    "coder": "agents/coder/graph.py:graph",
    "reviewer": "agents/reviewer/graph.py:graph",
}
backend.deploy(SERVED_AGENTS)
backend.up()
print(backend.status())


 Image open-langgraph-open-langgraph Building 


#1 [internal] load local bake definitions
#1 reading from stdin 618B done
#1 DONE 0.0s



#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.76kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.13-slim-bookworm


#3 DONE 0.8s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.1s



#5 [base 1/3] FROM docker.io/library/python:3.13-slim-bookworm@sha256:05b95397cac02b060ff1251afaa78087d92d7034369afbc8eb765631cada8257
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 1.34MB 0.1s done
#6 DONE 0.1s

#7 [base 3/3] RUN addgroup --system app && adduser --system --ingroup app app
#7 CACHED

#8 [builder 3/5] COPY vendor/open-langgraph-platform/README.md ./
#8 CACHED

#9 [final 1/9] RUN apt-get update && apt-get install -y --no-install-recommends     ca-certificates curl     && rm -rf /var/lib/apt/lists/*
#9 CACHED

#10 [final 2/9] COPY --from=builder /usr/local/lib/python3.13/site-packages/ /usr/local/lib/python3.13/site-packages/
#10 CACHED

#11 [base 2/3] WORKDIR /app
#11 CACHED

#12 [final 5/9] COPY vendor/open-langgraph-platform/alembic/ ./alembic/
#12 CACHED

#13 [builder 2/5] COPY vendor/open-langgraph-platform/pyproject.toml ./
#13 CACHED

#14 [builder 4/5] COPY vendor/open-langgraph-platform/src/ ./src/
#14 CACHED

#15 [final 6/9] COPY vendor/

...

 Container open-langgraph-open-langgraph-1 Created 
 Container open-langgraph-redis-1 Starting 
 Container open-langgraph-postgres-1 Starting 


 Container open-langgraph-redis-1 Started 
 Container open-langgraph-postgres-1 Started 
 Container open-langgraph-postgres-1 Waiting 
 Container open-langgraph-redis-1 Waiting 


 Container open-langgraph-redis-1 Healthy 
 Container open-langgraph-postgres-1 Healthy 
 Container open-langgraph-open-langgraph-1 Starting 


NAME                              IMAGE                           COMMAND                  SERVICE          CREATED         STATUS                                     PORTS
open-langgraph-open-langgraph-1   open-langgraph-open-langgraph   "sh -c 'alembic upgr…"   open-langgraph   6 seconds ago   Up Less than a second (health: starting)   0.0.0.0:8001->8000/tcp, [::]:8001->8000/tcp
open-langgraph-postgres-1         postgres:17-alpine              "docker-entrypoint.s…"   postgres         6 seconds ago   Up 5 seconds (healthy)                     0.0.0.0:6000->5432/tcp, [::]:6000->5432/tcp
open-langgraph-redis-1            redis:7-alpine                  "docker-entrypoint.s…"   redis            6 seconds ago   Up 5 seconds (healthy)                     0.0.0.0:6380->6379/tcp, [::]:6380->6379/tcp


 Container open-langgraph-open-langgraph-1 Started 


## 2. List registered assistants

In [3]:
import time

client = get_sync_client(url=backend.base_url)

for _attempt in range(10):
    try:
        assistants = list(client.assistants.search())
        if assistants:
            break
    except Exception as exc:
        print("waiting for backend...", exc)
    time.sleep(3)

for a in assistants:
    print(a["assistant_id"], "graph_id=" + str(a.get("graph_id")))


waiting for backend... [Errno 54] Connection reset by peer


waiting for backend... [Errno 54] Connection reset by peer


## 3. Call each agent directly

In [4]:
def call(name: str, message: str) -> str:
    last = None
    stream = client.runs.stream(
        None, name, input={"messages": [{"role": "user", "content": message}]}, stream_mode="values"
    )
    for chunk in stream:
        if chunk.event == "values" and chunk.data.get("messages"):
            last = chunk.data["messages"][-1]["content"]
    return last


print(call("coder", "What is 13 * 7? Just the number."))


91


In [5]:
print(call("researcher", "What is LangGraph in one sentence?"))


LangGraph is an open-source framework within the LangChain ecosystem that allows developers to build, manage, and execute complex AI agent workflows as controllable, stateful graphs.


In [6]:
print(call("reviewer", "Review this code: def add(a, b): return a + b"))


The code is correct and follows Python style conventions.

**Verdict:** Passes review.


## 4. Run the supervisor pipeline via RemoteGraph

The supervisor never imports `researcher`/`coder`/`reviewer` directly -- it only knows the backend's base URL and calls them through `langgraph.pregel.remote.RemoteGraph`, exactly as it would for a real LangGraph Platform deployment.

In [7]:
import os

os.environ["REMOTEGRAPH_BASE_URL"] = backend.base_url

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")
from agents.supervisor.graph import graph as supervisor_graph

task = "Write a one-line python function that returns the square of a number."
result = supervisor_graph.invoke({"task": task})
print("--- research ---")
print(result["research"])
print()
print("--- code ---")
print(result["code"])
print()
print("--- review ---")
print(result["review"])


--- research ---
```python
def square(n): return n * n
```

*(Alternatively, using the exponentiation operator: `def square(n): return n ** 2`)*

--- code ---
```python
square = lambda n: n * n
```

--- review ---
**Verdict: Correct.**

The code is syntactically correct, executes as expected, and uses a concise, idiomatic Python lambda function to map the squaring operation.

**Details:**
*   **Correctness:** The function correctly calculates $n^2$ for any input `n`.
*   **Style:** Using a lambda function is idiomatic for simple, single-expression operations like this.
*   **Conciseness:** The definition is minimal and readable for experienced Python users.


## 5. Tear down

In [8]:
backend.down()
print(backend.status())


 Container open-langgraph-open-langgraph-1 Stopping 


 Container open-langgraph-open-langgraph-1 Stopped 
 Container open-langgraph-open-langgraph-1 Removing 
 Container open-langgraph-open-langgraph-1 Removed 
 Container open-langgraph-postgres-1 Stopping 
 Container open-langgraph-redis-1 Stopping 


 Container open-langgraph-redis-1 Stopped 
 Container open-langgraph-redis-1 Removing 
 Container open-langgraph-redis-1 Removed 
 Container open-langgraph-postgres-1 Stopped 
 Container open-langgraph-postgres-1 Removing 
 Container open-langgraph-postgres-1 Removed 
 Network open-langgraph_default Removing 
 Network open-langgraph_default Removed 


NAME      IMAGE     COMMAND   SERVICE   CREATED   STATUS    PORTS
